# Unit 4 — Naive Bayes Classification
### Titanic Survival Analysis | Abrar Jawad | June 2026

---

## Introduction

Unit 3 used KNN to classify Titanic passengers. KNN worked by measuring
Euclidean distance — a passenger's predicted class was determined by who
their nearest neighbors were in feature space. It had no concept of which
features mattered more. `Sex` and `SibSp` contributed equally to the
distance calculation regardless of their actual predictive power.

Unit 4 introduces **Naive Bayes** — the first algorithm in this course
that works entirely from probability. Instead of asking *"who are your
nearest neighbors?"*, it asks *"given what we know about you, how
probable is each class?"*

The math is Bayes' theorem — already covered in the Statistics course:
P(A | B) = P(B | A) × P(A) / P(B)

Applied to Titanic:
P(Survived | features) ∝ P(features | Survived) × P(Survived)

The model computes this score for every class and predicts whichever
is higher. The "naive" assumption that makes this tractable: every
feature is treated as **independent given the class**. This is rarely
true in reality — Fare and Pclass are correlated, Sex and Age interact
— but the algorithm performs well despite it.

**Variant used:** `GaussianNB` — assumes continuous features follow a
normal distribution within each class. Mean and standard deviation are
computed per feature per class during `.fit()`. The same normal
distribution curve from the Statistics course, used as a probability
engine.

**Key difference from KNN:**
| | KNN | Naive Bayes |
|--|-----|-------------|
| Core mechanism | Euclidean distance | Bayes' theorem |
| `.fit()` stores | Entire training set | Means, variances, priors |
| Needs scaling | Yes | No |
| Feature importance | Blind | Visible via means table |
| Probability output | Weak | Yes, well-defined |

No `StandardScaler` is used in this notebook. Scaling was necessary for
KNN because large numerical ranges dominated the distance calculation.
Naive Bayes computes probabilities per feature independently — each
feature's scale is absorbed into its own Gaussian distribution. Scale
does not distort the result.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn import metrics

In [ ]:
df = pd.read_csv('../data/raw/train.csv')

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Fare']
df_model = df[features + ['Survived']].copy()

df_model['Sex'] = df_model['Sex'].map({'male': 0, 'female': 1})
df_model['Age'] = df_model['Age'].fillna(df_model['Age'].median())
df_model.dropna(inplace=True)

X = df_model[features]
y = df_model['Survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Training the Model


In [ ]:
gnb = GaussianNB()
gnb.fit(X_train, y_train)

In [ ]:
print("Class priors:", gnb.class_prior_)

print("Feature means per class:\n", gnb.theta_)

In [ ]:
y_pred = gnb.predict(X_test)

accuracy = metrics.accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

cm = metrics.confusion_matrix(y_test, y_pred)
print(cm)

print(metrics.classification_report(y_test, y_pred,
      target_names=['Did not survive', 'Survived']))

In [ ]:
# Confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did not survive', 'Survived'],
            yticklabels=['Did not survive', 'Survived'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Naive Bayes Confusion Matrix')
plt.tight_layout()
plt.savefig('../visuals/nb_confusion_matrix.png', dpi=150)
plt.show()

## Probability Outputs

`predict()` returns a hard label — 0 or 1. `predict_proba()` returns
the actual posterior probabilities behind that decision. Each row is
one passenger; column 0 is P(Did not survive), column 1 is P(Survived).

`predict()` is `predict_proba()` with a 0.5 threshold applied —
whichever column is higher becomes the predicted label.

This is a capability KNN does not have cleanly. Because Naive Bayes
computes explicit probabilities via Bayes' theorem, its confidence
estimates are well-defined. This matters in real systems where you need
not just a prediction but a measure of certainty — a medical model
saying "92% probability of malignancy" is more actionable than one
saying "malignant."

**What to look for in the confidence distribution plot:**
A well-calibrated model pushes predictions toward 0 or 1 — it is
decisive. A model with high uncertainty clusters near 0.5. Naive Bayes
tends toward overconfidence due to the independence assumption: when
multiple features all point toward the same class, their probabilities
multiply together and the product approaches the extremes quickly.

In [ ]:
y_prob = gnb.predict_proba(X_test)
print(y_prob[:5])

In [ ]:
# Visualize the confidence distribution
plt.figure(figsize=(8, 4))
plt.hist(y_prob[:, 1], bins=20, color='steelblue', edgecolor='white')
plt.axvline(0.5, color='red', linestyle='--', label='Decision threshold')
plt.xlabel('Predicted probability of survival')
plt.ylabel('Number of passengers')
plt.title('Naive Bayes — Confidence Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('../visuals/nb_probability_distribution.png', dpi=150)
plt.show()

In [ ]:
feature_names = ['Pclass', 'Sex', 'Age', 'SibSp', 'Fare']

print("=== Class Priors ===")
for i, cls in enumerate(['Did not survive', 'Survived']):
    print(f"  P({cls}) = {gnb.class_prior_[i]:.3f}")

print("\n=== Feature Means per Class ===")
means_df = pd.DataFrame(
    gnb.theta_,
    columns=feature_names,
    index=['Did not survive', 'Survived']
)
print(means_df.round(3))

## Conclusion

### Results

| Model | Accuracy | Recall (Survived) | FN |
|-------|----------|-------------------|----|
| Naive baseline | 61.5% | — | — |
| KNN K=7 | 81.6% | 68% | 24 |
| Naive Bayes | 76.5% | 72% | 21 |

Naive Bayes scores lower on overall accuracy but finds 3 more actual
survivors than KNN — FN drops from 24 to 21. Which model is "better"
depends on what the model is used for. In a rescue prioritization
system, missing a survivor (FN) is more costly than a false alarm (FP).
Accuracy alone would select KNN. The confusion matrix tells a more
nuanced story.

### What Naive Bayes Does That KNN Cannot

The means table (`gnb.theta_`) makes the model's learned knowledge
readable. The gap in `Sex` means (0.144 vs 0.675) is the largest in
the table — the model has quantified that Sex is the strongest
predictor, something KNN was blind to entirely.

The confidence distribution shows the overconfidence signature of the
independence assumption: predictions cluster near 0 and 1 rather than
distributing smoothly. When multiple features point toward the same
class, their probabilities multiply and the product compounds toward the
extremes.

### Limitations

The independence assumption is violated by this dataset. `Fare` and
`Pclass` are correlated — a 1st class ticket costs more, so these
features carry redundant information. When both are included, their
overlapping signal is double-counted. A model that accounts for feature
interactions would handle this more accurately.

This is exactly what Unit 5 addresses. Decision Trees learn explicit
if-else rules that capture interactions between features — *"if female
AND 1st class, predict survived"* — without assuming independence.
Random Forests extend this by averaging hundreds of trees to reduce
overfitting. Unit 5 also introduces feature importances: a ranked,
quantitative measure of which features drove predictions — the cleanest
version of what the means table approximates here.

### Key Concepts — Unit 4

| Concept | Key point |
|---------|-----------|
| Bayes' theorem | P(class \| features) ∝ P(features \| class) × P(class) |
| Prior | P(Survived) — class proportions in training data |
| Likelihood | P(feature \| class) — Gaussian per feature per class |
| Naive assumption | Features independent given class — rarely true, works anyway |
| GaussianNB | Continuous features modeled as normal distributions |
| No scaling needed | Probabilities per feature — scale absorbed into Gaussian |
| predict_proba() | Posterior probabilities, not just hard labels |
| Overconfidence | Independence assumption compounds probabilities toward extremes |
| gnb.theta_ | Means table — the model's full learned knowledge, readable |
| Accuracy vs recall | Metric choice depends on the cost structure of errors |